In [1]:
suppressMessages(suppressWarnings({
    library(ggplot2)
    library(igraph)
    library(tidyverse)
    library(ggpubr)
}))

In [2]:
# load wilcox test results and plot barplot to show celltype non-speicfic dominate expression across species
species <- c("Hsap", "Mmus", "Pvit", "Pmar")
type <- c("WGD", "SSD")
mtx <- c("exp", "pct")

pairwise_res <- Reduce(rbind, lapply(species, FUN = function(s){
    Reduce(rbind, lapply(type, FUN = function(t){
        Reduce(rbind, lapply(mtx, FUN = function(m){
            info <- read.delim(paste0(s,"/",m,"/",s, ".", m, ".", t, ".pairwise_wilcox_test.txt"), header = T)
            info$species <- s
            info$type <- t
            info$mtx <- m
            return(info)
        }))
    }))
}))

In [3]:
head(pairwise_res)

,group1,group2,n1,n2,statistic,p,p.adj,p.adj.signif,species,type,mtx
,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
1,ENSG00000160539,ENSG00000205808,17,17,140,0.00134,NA,,Hsap,WGD,exp
2,ENSG00000018625,ENSG00000074370,17,17,105,0.05900,1,ns,Hsap,WGD,exp
3,ENSG00000018625,ENSG00000163399,17,17,50,0.22500,1,ns,Hsap,WGD,exp
4,ENSG00000018625,ENSG00000070961,17,17,45,0.14500,1,ns,Hsap,WGD,exp
5,ENSG00000018625,ENSG00000187527,17,17,118,0.05100,1,ns,Hsap,WGD,exp
6,ENSG00000018625,ENSG00000067842,17,17,74,0.92700,1,ns,Hsap,WGD,exp


In [4]:
# input a single (SSD)paralog/ohnolog family, 
# return the number of paralogs and number of pairs having significant difference
get_sign_res <- function(diff_, pairs){
    if(length(pairs) == 2){
        n_sign <- diff_[diff_$group1 %in% pairs, "p"]
    } else if (length(pairs) > 2) {
        n_sign <- diff_[diff_$group1 %in% pairs, "p.adj"] #p.adj.signif
    } else {
        n_sign <- NA
    }
    n_sign <- n_sign[!is.na(n_sign) & n_sign < 0.01]
    info <- c(length(pairs), length(n_sign))
}

In [5]:
res <- Reduce(rbind, lapply(species, FUN = function(s){
    Reduce(rbind, lapply(type, FUN = function(t){
        Reduce(rbind, lapply(mtx, FUN = function(m){
            
            tmp <- pairwise_res %>% filter(species == s & type == t & mtx == m)
            g <- graph_from_data_frame(tmp[,c(1,2)], directed = FALSE)
            components <- components(g)$membership
            family <- split(names(components), components)
            
            res <- Reduce(rbind, lapply(seq_along(family), FUN = function(x){
                pairs = family[[x]]
                get_sign_res(tmp, pairs)
            }))
            res <- data.frame(res)
            colnames(res) <- c("number_of_paralogs", "number_of_sign")
            res$species <- s
            res$type <- t
            res$mtx <- m
            return(res)
        }))
    }))
}))

In [6]:
head(res)

,number_of_paralogs,number_of_sign,species,type,mtx
,<int>,<int>,<chr>,<chr>,<chr>
init,2,1,Hsap,WGD,exp
X,19,67,Hsap,WGD,exp
X.1,2,1,Hsap,WGD,exp
X.2,4,3,Hsap,WGD,exp
X.3,4,2,Hsap,WGD,exp
X.4,10,3,Hsap,WGD,exp


In [7]:
plot_res <- res %>% group_by(species, type, mtx) %>% 
  summarise(
      number_all = n(),
      number_all_sign = sum(number_of_sign > 0),
      number_all_insign = number_all - number_all_sign,
      number2 = sum(number_of_paralogs == 2),
      number2_sign = sum(number_of_paralogs == 2 & number_of_sign > 0),
      number2_insign = number2 - number2_sign,
      number4 = sum(number_of_paralogs <= 4),
      number4_sign = sum(number_of_paralogs <= 4 & number_of_sign > 0),
      number4_insign = number4 - number4_sign
  )

`summarise()` has grouped output by 'species', 'type'. You can override using
the `.groups` argument.


In [8]:
plot_res1 <- plot_res %>% pivot_longer(cols = c(number_all_sign, number_all_insign), 
               names_to = "variable", 
               values_to = "value")
plot_res1$variable <- factor(plot_res1$variable, levels = c("number_all_sign", "number_all_insign"))
plot_res1$type <- factor(plot_res1$type, levels = c("WGD", "SSD"))
plot_res1$species <- factor(plot_res1$species, levels = species)

In [9]:
pdf("summary_family.exp.significant_diff.pdf", width = 5, height = 4)
plot_res1 %>% filter(mtx == "exp") %>% ggbarplot(x = "variable", y = "value", color = "variable", fill = "variable") + 
    scale_fill_manual(values=c("#99990066","#66666666"))+
    scale_color_manual(values=c("#999900","#666666")) +
    facet_grid(vars(type), vars(species)) 
dev.off()

pdf 
  2

In [10]:
pdf("summary_family.pct.significant_diff.pdf", width = 5, height = 4)
plot_res1 %>% filter(mtx == "pct") %>% ggbarplot(x = "variable", y = "value", color = "variable", fill = "variable") + 
    scale_fill_manual(values=c("#99990066","#66666666"))+
    scale_color_manual(values=c("#999900","#666666")) +
    facet_grid(vars(type), vars(species)) 
dev.off()

pdf 
  2

In [11]:
# since for family member >=3, we need multiple comparison to confirm a dominant expression
# in this dominant plot, I only plot familt with 2 copies
plot_res2 <- plot_res %>% pivot_longer(cols = c(number2_sign, number2_insign), 
               names_to = "variable", 
               values_to = "value")
plot_res2$variable = factor(plot_res2$variable, levels = c("number2_sign", "number2_insign"))
plot_res2$type = factor(plot_res2$type, levels = c("WGD", "SSD"))
plot_res2$species <- factor(plot_res2$species, levels = species)

pdf("summary_family.exp.dominant_copy_at_family_containing_2copies.pdf", width = 5, height = 4)
plot_res2 %>% filter(mtx == "exp") %>% ggbarplot(x = "variable", y = "value", color = "variable", fill = "variable") + 
    scale_fill_manual(values=c("#99990066","#66666666"))+
    scale_color_manual(values=c("#999900","#666666")) +
    facet_grid(vars(type), vars(species)) 
dev.off()

pdf("summary_family.pct.dominant_copy_at_family_containing_2copies.pdf", width = 5, height = 4)
plot_res2 %>% filter(mtx == "pct") %>% ggbarplot(x = "variable", y = "value", color = "variable", fill = "variable") + 
    scale_fill_manual(values=c("#99990066","#66666666"))+
    scale_color_manual(values=c("#999900","#666666")) +
    facet_grid(vars(type), vars(species)) 
dev.off()

pdf 
  2

pdf 
  2

In [19]:
# Around two-thirds (58-76%) of these paralogue families, 
# whether derived by WGD or SSD, have a significantly dominant copy in a pervasive way
plot_res2 %>% filter(variable == 'number2_sign') %>% group_by(species, type, mtx) %>% summarise(value/number2) %>% ungroup()

`summarise()` has grouped output by 'species', 'type'. You can override using
the `.groups` argument.


species,type,mtx,value/number2
<fct>,<fct>,<chr>,<dbl>
Hsap,WGD,exp,0.6500664
Hsap,WGD,pct,0.6586985
Hsap,SSD,exp,0.6075269
Hsap,SSD,pct,0.6137993
Mmus,WGD,exp,0.6243056
Mmus,WGD,pct,0.6291667
Mmus,SSD,exp,0.5824308
Mmus,SSD,pct,0.5872443
Pvit,WGD,exp,0.6194517
